# Credit Card Fraud Detection - EDA and Modeling

This notebook contains exploratory data analysis and modeling experiments for credit card fraud detection.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().absolute().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Load Data


In [ ]:
from src.data.load_data import load_data

# Load data (use sample_size=None for full dataset)
df = load_data(sample_size=10000)  # Use smaller sample for demo
print(f"Dataset shape: {df.shape}")
df.head()


## 2. Exploratory Data Analysis

### 2.1 Class Imbalance


In [ ]:
class_counts = df['Class'].value_counts()
class_props = df['Class'].value_counts(normalize=True)

print("Class Distribution:")
print(class_counts)
print("\nClass Proportions:")
print(class_props)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
class_counts.plot(kind='bar', color=['blue', 'red'])
plt.title('Class Distribution')
plt.xlabel('Class (0=Normal, 1=Fraud)')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
class_props.plot(kind='pie', autopct='%1.2f%%', colors=['blue', 'red'])
plt.title('Class Proportions')
plt.ylabel('')

plt.tight_layout()
plt.show()


### 2.2 Distribution of Amount


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Overall distribution
axes[0, 0].hist(df['Amount'], bins=50, edgecolor='black')
axes[0, 0].set_title('Distribution of Transaction Amount')
axes[0, 0].set_xlabel('Amount')
axes[0, 0].set_ylabel('Frequency')

# Log scale
axes[0, 1].hist(np.log1p(df['Amount']), bins=50, edgecolor='black')
axes[0, 1].set_title('Distribution of Log(Amount + 1)')
axes[0, 1].set_xlabel('Log(Amount + 1)')
axes[0, 1].set_ylabel('Frequency')

# By class
df[df['Class']==0]['Amount'].hist(bins=50, alpha=0.7, label='Normal', ax=axes[1, 0])
df[df['Class']==1]['Amount'].hist(bins=50, alpha=0.7, label='Fraud', ax=axes[1, 0])
axes[1, 0].set_title('Amount Distribution by Class')
axes[1, 0].set_xlabel('Amount')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# Box plot
df.boxplot(column='Amount', by='Class', ax=axes[1, 1])
axes[1, 1].set_title('Amount by Class')
axes[1, 1].set_xlabel('Class')
axes[1, 1].set_ylabel('Amount')

plt.tight_layout()
plt.show()

print(f"\nAmount Statistics:")
print(df.groupby('Class')['Amount'].describe())


### 2.3 Distribution of Time


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Time distribution
axes[0].hist(df['Time'], bins=50, edgecolor='black')
axes[0].set_title('Distribution of Time')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Frequency')

# Time by class
df[df['Class']==0]['Time'].hist(bins=50, alpha=0.7, label='Normal', ax=axes[1])
df[df['Class']==1]['Time'].hist(bins=50, alpha=0.7, label='Fraud', ax=axes[1])
axes[1].set_title('Time Distribution by Class')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()


### 2.4 PCA Features Distribution


In [ ]:
# Sample a few V features for visualization
v_features = [f'V{i+1}' for i in range(min(6, 28))]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(v_features):
    df[df['Class']==0][feature].hist(bins=30, alpha=0.7, label='Normal', ax=axes[idx])
    df[df['Class']==1][feature].hist(bins=30, alpha=0.7, label='Fraud', ax=axes[idx])
    axes[idx].set_title(f'{feature} Distribution by Class')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()

plt.tight_layout()
plt.show()


### 2.5 Correlation Heatmap


In [ ]:
# Calculate correlation with target
correlations = df.corr()['Class'].sort_values(ascending=False)
print("Top 10 Features Correlated with Class:")
print(correlations.head(11))  # Include Class itself

# Correlation heatmap for selected features
selected_features = ['Time', 'Amount'] + [f'V{i+1}' for i in range(10)] + ['Class']
corr_matrix = df[selected_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (Selected Features)')
plt.tight_layout()
plt.show()


## 3. Modeling Experiments

### 3.1 Prepare Data


In [ ]:
from src.data.preprocess import preprocess_data

X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocess
X_train_processed, X_test_processed, preprocessor = preprocess_data(
    X_train, X_test, scale_type='robust', use_pca=False
)

print(f"Training set: {X_train_processed.shape}")
print(f"Test set: {X_test_processed.shape}")


### 3.2 Baseline Model (No Resampling)


In [ ]:
from src.models.train import train_logistic_regression
from src.models.evaluate import evaluate_model

# Train baseline model
model_baseline = train_logistic_regression(
    X_train_processed, y_train, class_weight='balanced'
)

# Evaluate
metrics_baseline = evaluate_model(model_baseline, X_test_processed, y_test, "Baseline")
print("Baseline Model Metrics:")
for key, value in metrics_baseline.items():
    print(f"  {key}: {value:.4f}")


### 3.3 Model with SMOTE Oversampling


In [ ]:
from src.models.train import apply_smote

# Apply SMOTE
X_train_smote, y_train_smote = apply_smote(X_train_processed, y_train)
print(f"After SMOTE - Training set: {X_train_smote.shape}")
print(f"Class distribution: {pd.Series(y_train_smote).value_counts()}")

# Train model with SMOTE
model_smote = train_logistic_regression(
    X_train_smote, y_train_smote, class_weight=None
)

# Evaluate
metrics_smote = evaluate_model(model_smote, X_test_processed, y_test, "SMOTE")
print("\nSMOTE Model Metrics:")
for key, value in metrics_smote.items():
    print(f"  {key}: {value:.4f}")


### 3.4 Model with Random Undersampling


In [ ]:
from imblearn.under_sampling import RandomUnderSampler

# Apply undersampling
undersampler = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = undersampler.fit_resample(X_train_processed, y_train)
print(f"After Undersampling - Training set: {X_train_under.shape}")
print(f"Class distribution: {pd.Series(y_train_under).value_counts()}")

# Train model with undersampling
model_under = train_logistic_regression(
    X_train_under, y_train_under, class_weight=None
)

# Evaluate
metrics_under = evaluate_model(model_under, X_test_processed, y_test, "Undersampling")
print("\nUndersampling Model Metrics:")
for key, value in metrics_under.items():
    print(f"  {key}: {value:.4f}")


### 3.5 Comparison of Resampling Strategies


In [ ]:
# Compare all strategies
comparison = pd.DataFrame({
    'Baseline (Class Weight)': metrics_baseline,
    'SMOTE': metrics_smote,
    'Undersampling': metrics_under
}).T

print("\nModel Comparison:")
print(comparison.round(4))

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metrics_to_plot = ['precision', 'recall', 'f1', 'roc_auc']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    comparison[metric].plot(kind='bar', ax=ax, color=['blue', 'green', 'orange'])
    ax.set_title(f'{metric.upper()} Comparison')
    ax.set_ylabel(metric)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
